In [1]:
import os

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, StratifiedKFold


In [2]:
def read_and_info(filepath):
    """Load a Stata file, print path + shape + first rows (as in BA Q1)."""
    df = pd.read_stata(filepath, convert_categoricals=False)
    print(f"{filepath}: {df.shape[0]} rows, {df.shape[1]} columns")
    display(df.head().iloc[:, :8])
    return df

main   = read_and_info("../data/raw/coding_suseti_pmt.dta")
labels = read_and_info("../data/raw/mistargeting_CORRECTED.dta")

# 1. Check the merge keys before merging (BA Q5.1)
print("hhid unique in main:          ", main["hhid"].is_unique) #hhid is the household id - which is in both datasets -> to merge this must first ascertain that its unique
print("hhid_suseti unique in labels: ", labels["hhid_suseti"].is_unique)
assert main["hhid"].is_unique and labels["hhid_suseti"].is_unique


../data/raw/coding_suseti_pmt.dta: 5756 rows, 347 columns


,hhid,hhmale,hhage,hhmarried,hhage2,hhmalemarr,hheduc1,hheduc2
0,001025,1,80.0,1,6400.0,1,1,0
1,001302,1,58.0,1,3364.0,1,0,0
2,001468,1,35.0,1,1225.0,1,0,0
3,001473,1,68.0,1,4624.0,1,0,1
4,001476,0,78.0,0,6084.0,0,1,0


../data/raw/mistargeting_CORRECTED.dta: 5756 rows, 95 columns


,hhea,avg_CONSUMPTION,avg_EDUCATIONHHHEAD,avg_PMTSCORE,SHAREAGRICULTURALHH,avg_YREDUCATIONHHHEAD,EDUCATIONRTHEAD,YREDUCATIONRTHEAD
0,1,526.538879,2.777778,12.664343,100.0,6.333333,3,6
1,1,526.538879,2.777778,12.664343,100.0,6.333333,3,6
2,1,526.538879,2.777778,12.664343,100.0,6.333333,3,6
3,1,526.538879,2.777778,12.664343,100.0,6.333333,3,6
4,1,526.538879,2.777778,12.664343,100.0,6.333333,3,6


hhid unique in main:           True
hhid_suseti unique in labels:  True


In [3]:
# 2. Reduce both datasets to the columns we need.
# We list what we KEEP (not what we drop): any column not listed is removed,
# so no leakage column can slip through by being forgotten in a drop list.

# --- Columns that are not features: IDs, grouping, quota, baseline, targets, filter helpers
id_cols       = ["hhid", "hhea", "kabu", "prop"]   # household ID, village, district, province
quota_cols    = ["share"]                           # village quota for the selection rule
baseline_cols = ["pcexppred"]   #expected consumption by government - government formula score (baseline only, never a feature)
target_cols   = ["LOGCONSUMPTION"]                  # regression target (poor comes from labels)
helper_cols   = ["age1564",                         # only to build no_workingage, dropped after
                 "hheduc1"]                         # only to find heads with missing education (all 4 hheduc = 0), dropped after

# --- Set A: the 62 variables of the government PMT formula (those with a c_* coefficient)
set_a_dwelling  = ["pcfloor", "tfloor", "twall", "troof", "toilet", "water", "lighting", "fcook", "house"]
set_a_household = ["hhsize", "hhsize2", "age04", "eschild", "jschild", "sschild", "depratio"]
set_a_head      = ["hhage", "hhage2", "hhmale", "hhmarried", "hhmalemarr",
                   "hheduc2", "hheduc3", "hheduc4", "hhsector1", "hhsector2", "hhsector3",
                   "formal", "informal"]
set_a_educ      = ["higheduc2", "higheduc3", "higheduc4"]
set_a_credit    = ["credit"]                        # self-reported, but in the government formula
set_a_location  = ["urban"]
set_a_village   = ["vill2dist", "villsd", "villsmp", "villpusk", "villpolin", "villposy",
                   "villdokt", "villbidan", "villrtype", "villpmplace", "villcred"]
set_a_assets    = ["se2_17", "se2_11_12", "se2_125", "se2_126_127", "se2_119", "se2_117",
                   "se2_115", "se2_13_15", "se2_14", "se2_114", "se2_18", "se2_124",
                    "se2_122_123", "se5_110", "se5_11_12",
                   "se5_15_16", "se2_138"]          # goat/sheep and agricultural land are in the formula too
set_a = (set_a_dwelling + set_a_household + set_a_head + set_a_educ + set_a_credit
         + set_a_location + set_a_village + set_a_assets)

# --- Set B: extra items an enumerator can see or verify
set_b_assets = ["se2_16", "se2_111", "se2_110", "se2_118", "se2_112", "se2_121",
                "se2_132_133", "se2_128", "se2_129", "se5_17", "se5_13", "se2_139",
                "FLOORTOPQUALITY"]
# hr0607, hr0608: not kept. The diagnosis (step 5) showed they have no variable label
# (only "07 hr06", "08 hr06") and no matching 0/1 dummy, so we cannot say which item they count.
unlabelled_counts = ["hr0607", "hr0608"]
set_b_counts = [c for c in main.columns if c.startswith("hr06") and c not in unlabelled_counts]   # number of each asset owned
set_b = set_b_assets + set_b_counts

main_cols   = id_cols + quota_cols + baseline_cols + target_cols + helper_cols + set_a + set_b
labels_cols = ["hhid_suseti", "poor", "povline_poor", "RTHEAD"]   # key, target, its line, filter

# Checks on the lists themselves
assert len(set_a) == 62, f"set A should have 62 variables, has {len(set_a)}"
assert len(main_cols) == len(set(main_cols)), "a column is listed twice"
missing = [c for c in main_cols if c not in main.columns] + [c for c in labels_cols if c not in labels.columns]
assert not missing, f"columns not found: {missing}"

# Reduce: everything not listed above is dropped
print("before:", main.shape, labels.shape)
main   = main[main_cols].copy()
labels = labels[labels_cols].copy()
print("after: ", main.shape, labels.shape)

# 3. Leakage check: nothing derived from consumption, the government score,
# the allocation or self-reports survived the reduction
# consumption related varianbles contain the answer to what we want to predict so drop it
forbidden_prefixes = ("c_", "comp", "temp", "rank", "nhhrank", "RTS", "MISTARGET", "PMT", "RANK", "avg_")
forbidden_cols = ["CONSUMPTION", "hhconsumption", "wfood", "mfood", "mnonfood", "lnpcexpe", "epcexpe",
                  "povertygap", "verypoor", "healthcare", "shock", "maintreatment", "quota", "quota_final"]
leaks = [c for c in list(main.columns) + list(labels.columns)
         if c.startswith(forbidden_prefixes) or c in forbidden_cols or c.endswith(("_avg", "_dm"))]
assert not leaks, f"leakage columns still present: {leaks}"
print("leakage check passed")

before: (5756, 347) (5756, 95)
after:  (5756, 110) (5756, 4)
leakage check passed


In [4]:
# 4. Give every column the data type that matches what it represents (as in BA Q2).
#    Types: text -> "string", whole numbers -> "Int64" (nullable: keeps missing values as <NA>),
#    measurements with decimals -> "float64".
#    Many columns were read as int8 (max 127) or float32 (less precise): converting them to
#    Int64 / float64 avoids silent overflow later (e.g. 12**2 = 144 does not fit in int8).

# --- 1. Before: how pandas read the columns
print("before:", main.dtypes.astype(str).value_counts().to_dict(), "|", labels.dtypes.astype(str).to_dict())

# --- 2. Schema: which type each group of columns should have
id_text   = ["hhid", "hhid_suseti"]                 # identifiers: labels, not numbers (keep leading zeros "001025")
id_codes  = ["hhea", "kabu", "prop"]                # village / district / province codes: whole numbers, not measurements
measures  = ["share", "pcexppred", "LOGCONSUMPTION", "pcfloor", "depratio",
             "vill2dist", "povline_poor"]           # measurements with decimals
counts    = ["hhsize", "hhsize2", "age04", "eschild", "jschild", "sschild",
             "age1564", "hhage", "hhage2"]          # whole-number counts (age in years, age squared)
hr06_whole   = [c for c in set_b_counts if c != "hr0627"]   # number of each asset owned: whole numbers - this 
hr06_decimal = ["hr0627"]                                   # jewelry: has values like 0.5 (likely grams) -> keep decimals
binary    = [c for c in set_a + set_b_assets + ["hheduc1"]
             if c not in measures + counts]         # 0/1 dummies (se2_139 is still coded 1/3; fixed in the cleaning step)
labels_codes = ["poor", "RTHEAD"]                   # 0/1 target and filter flag

desired_dtypes = {}
desired_dtypes.update({c: "string"  for c in id_text})
desired_dtypes.update({c: "Int64"   for c in id_codes + counts + hr06_whole + binary + labels_codes})
desired_dtypes.update({c: "float64" for c in measures + hr06_decimal})

# Every kept column must have exactly one type
all_cols = list(main.columns) + list(labels.columns)
assert set(desired_dtypes) == set(all_cols), \
    f"no type for: {set(all_cols) - set(desired_dtypes)} | not in data: {set(desired_dtypes) - set(all_cols)}"

# --- 3. Apply the schema with one loop over both datasets
for df in (main, labels):
    for col, t in desired_dtypes.items():
        if col in df.columns:
            df[col] = df[col].astype(t)

# --- 4. After: verify
print("after: ", main.dtypes.astype(str).value_counts().to_dict(), "|", labels.dtypes.astype(str).to_dict())
print("hhid examples (leading zeros kept):", main["hhid"].head(3).tolist())
assert (main["hhsize"] ** 2 == main["hhsize2"]).all(), "hhsize2 should equal hhsize squared"   # failed with int8

before: {'int8': 63, 'float64': 36, 'float32': 7, 'int16': 3, 'str': 1} | {'hhid_suseti': 'str', 'poor': 'float64', 'povline_poor': 'float32', 'RTHEAD': 'int8'}
after:  {'Int64': 102, 'float64': 7, 'string': 1} | {'hhid_suseti': 'string', 'poor': 'Int64', 'povline_poor': 'float64', 'RTHEAD': 'Int64'}
hhid examples (leading zeros kept): ['001025', '001302', '001468']


In [5]:
# 5. Diagnose (BA Q3.1): look for problems in each dataset BEFORE fixing anything.
#    Nothing is changed in this cell; it only prints what the cleaning step has to deal with.

# --- 1. Missing values per column (only columns with at least one missing value)
for name, data in [("main", main), ("labels", labels)]:
    na = data.isna().sum() #gives the number of missing values each column
    na = na[na > 0].sort_values(ascending=False) #keeps only columns which at least one missing value and sorts from highest to smallest
    print(f"\n{name} {data.shape}: {len(na)} columns with missing values") #makes it into a list
    print(na.to_string())

# --- 2. Duplicated identifiers and households per village
print("\nduplicated hhid in main:          ", main["hhid"].duplicated().sum()) #check that the identifier (household number) is only once for each
print("duplicated hhid_suseti in labels: ", labels["hhid_suseti"].duplicated().sum())
print("households per village:           ", main.groupby("hhea").size().value_counts().to_dict()) #counts how many households per village there are
print("subvillage heads per village:     ", labels.merge(main[["hhid", "hhea"]], left_on="hhid_suseti", right_on="hhid")
                                                 .groupby("hhea")["RTHEAD"].sum().value_counts().to_dict())

# --- 3. Implausible values: things that cannot be true or contradict each other
binary_cols = binary + ["poor", "RTHEAD"]
checks = {
    # ranges - this checks columns seperately
    "hhsize < 1":                           (main["hhsize"] < 1).sum(), #these always count how many of these fall in this unusual/alarming range
    "hhage < 15 or > 100":                  ((main["hhage"] < 15) | (main["hhage"] > 100)).sum(),
    "pcfloor <= 0":                         (main["pcfloor"] <= 0).sum(),
    "pcfloor > 200 m² per person":          (main["pcfloor"] > 200).sum(),
    "depratio < 0":                         (main["depratio"] < 0).sum(),
    "vill2dist < 0":                       (main["vill2dist"] < 0).sum(),
    "share outside (0, 1]":                 ((main["share"] <= 0) | (main["share"] > 1)).sum(),
    "pcexppred < 0 (negative predicted consumption)": (main["pcexppred"] < 0).sum(),
    "asset count (hr06xx) < 0":            int((main[set_b_counts] < 0).sum().sum()),
    # coding
    "0/1 columns with other values":        [c for c in binary_cols
                                             if not set((main if c in main else labels)[c].dropna().unique()) <= {0, 1}], #checks that 0 1 columns are actually 01 columns and not with other numbers. take column from main or else from labes (poor and rthead), drop the na (this checked in other part) and lists distinct values - checks if those are 0 1 if not - keep it in a list
    # consistency between columns - this check columns together if it even makes sense
    "hhsize2 != hhsize²":                   (main["hhsize2"] != main["hhsize"] ** 2).sum(), #household size^2 should be the same as household size squared -> although these are strongly correlated not perfectly collinear and OLS can still calculate (inverse) thus we keep both
    "hhage2 != hhage²":                     (main["hhage2"] != main["hhage"] ** 2).sum(),
    "age04 > hhsize":                       (main["age04"] > main["hhsize"]).sum(), #check if some households have more children as household memebers
    "children in school > hhsize":          ((main["eschild"] + main["jschild"] + main["sschild"]) > main["hhsize"]).sum(),
    "head education missing (all 4 hheduc = 0)": (main[["hheduc1", "hheduc2", "hheduc3", "hheduc4"]].sum(axis=1) == 0).sum(), #this means that no highest education level was listed
    "head education: more than one hheduc = 1":  (main[["hheduc1", "hheduc2", "hheduc3", "hheduc4"]].sum(axis=1) > 1).sum(), #if larger as two more than 1 highest education level was listend - impossable
    "depratio missing & age1564 == 0":      (main["depratio"].isna() & (main["age1564"] == 0)).sum(),
}
print("\nimplausible / inconsistent values:")
for k, v in checks.items(): #k is the name and v the result
    print(f"  {k:48s} {v}")

# --- 4. Asset counts (hr06xx) vs. the matching 0/1 asset dummy
#     Expected: count missing <=> dummy == 0 (household does not own the item)
count_to_dummy = {"hr0609": "se2_17", "hr0610": "se2_16", "hr0611": "se2_111", "hr0612": "se2_110", #this maps together the dummy -- do they own yes or no and the number of how much they own
                  "hr0613": "se2_115", "hr0614": "se2_13_15", "hr0615": "se2_11_12", "hr0616": "se2_14",
                  "hr0617": "se2_114", "hr0618": "se2_117", "hr0619": "se2_119", "hr0620": "se2_118",
                  "hr0621": "se2_124", "hr0622": "se2_128", "hr0623": "se2_125", "hr0625": "se2_126_127",
                  "hr0626": "se2_129", "hr0627": "se2_122_123", "hr0628": "se5_110", "hr0629": "se5_17",
                  "hr0630": "se5_15_16", "hr0631": "se5_11_12", "hr0632": "se5_13", "hr0633": "se2_132_133",
                  "hr0634": "se2_112", "hr0635": "se2_121"}      # hr0607, hr0608: unlabelled, no dummy -> can't verify if NA should become 0 thus this is dropped
rows = []
for cnt, dum in count_to_dummy.items():
    rows.append({"count": cnt, "dummy": dum,
                 "count missing, dummy=0 (ok: none owned)":  (main[cnt].isna() & (main[dum] == 0)).sum(),
                 "count missing, dummy=1 (problem)":         (main[cnt].isna() & (main[dum] == 1)).sum(),
                 "count present, dummy=0 (contradiction)":   (main[cnt].notna() & (main[dum] == 0)).sum(),
                 "dummy missing, count missing":             (main[dum].isna() & main[cnt].isna()).sum(),
                 "dummy missing, count present":             (main[dum].isna() & main[cnt].notna()).sum()})
hr = pd.DataFrame(rows).set_index("count")
problem_cols = [c for c in hr.columns if c not in ("dummy", "count missing, dummy=0 (ok: none owned)")]
print("\nasset counts vs. dummies (only rows with a problem):")
print(hr[(hr[problem_cols] > 0).any(axis=1)].to_string())

# --- 5. Targets
print("\nLOGCONSUMPTION range:", main["LOGCONSUMPTION"].min().round(2), "to", main["LOGCONSUMPTION"].max().round(2)) #this checks if entries in log-consumption are plausible from smallest to highest
q1, q3 = main["LOGCONSUMPTION"].quantile([0.25, 0.75]) #then computes the smallest quantiles
print("Tukey outliers on log consumption (k=3):", #tukey fences: values more than 3x the range Q3-Q1 below Q1 and above Q3 count as extreme outliers (with k=3 as conservative outer fence)
      ((main["LOGCONSUMPTION"] < q1 - 3 * (q3 - q1)) | (main["LOGCONSUMPTION"] > q3 + 3 * (q3 - q1))).sum())
#does poor match the definition
chk = main[["hhid", "LOGCONSUMPTION"]].merge(labels, left_on="hhid", right_on="hhid_suseti") #temporary merge to check that consumption is in main dataset and poor and poverty line in labels --> check that these coincide
chk = chk[chk["poor"].notna()]
on_line = np.isclose(np.exp(chk["LOGCONSUMPTION"]), chk["povline_poor"], rtol=1e-6)   # consumption exactly = poverty line -> convert logconumption with exp back to rupiah - isclose maps those households that are exactly on poverty line except for some small rounding differences
below   = (np.exp(chk["LOGCONSUMPTION"]) < chk["povline_poor"]) & ~on_line
print("poor share:", round(chk["poor"].mean(), 3))
print("poor != (consumption below the line), away from the line:", ((below.astype(int) != chk["poor"]) & ~on_line).sum())
print("households exactly on the poverty line:", on_line.sum(),
      "| of these coded poor:", int(chk.loc[on_line, "poor"].sum()))


main (5756, 110): 40 columns with missing values
hr0626            5744
hr0622            5701
hr0632            5699
hr0613            5657
hr0633            5599
hr0625            5441
hr0618            5401
hr0629            5368
hr0630            5223
hr0620            5151
hr0631            5112
hr0634            4868
hr0617            4838
hr0635            4081
hr0611            3989
hr0609            3956
hr0614            3172
hr0612            3168
hr0628            3129
hr0616            3021
hr0621            3009
hr0610            2881
hr0627            2681
hr0623            2677
hr0619            1983
hr0615            1136
depratio           228
pcfloor             52
se2_139             13
toilet               8
hhage2               6
hhage                6
se2_122_123          4
LOGCONSUMPTION       3
se5_17               1
se2_128              1
se2_121              1
se2_112              1
se5_110              1
water                1

labels (5756, 4): 1 columns w

### Diagnosis: what we found (all 5,756 households, before any row is dropped)

<details>
<summary><b>Click to open: findings and what the cleaning step does about each</b></summary>

**No problem (checked and fine)**
- No duplicated IDs in either file; both merge keys are unique.
- Exactly **one subvillage head per village** (640 in total).
- `hhsize2 = hhsize²` and `hhage2 = hhage²` everywhere (after the type fix; with `int8` this check failed).
- No household has more young children or school children than members; no negative or zero sizes, distances or floor areas; `share` is between 0 and 1.
- `poor` matches "consumption below the poverty line" for every household away from the line.

**Things the cleaning step must handle**

| Finding | Count | Decision |
|---|---|---|
| Villages with only 8 instead of 9 households | 4 | Keep. Check "≤ 9 per village" instead of "exactly 9". |
| `LOGCONSUMPTION` and `poor` missing | 3 | Drop: no target, cannot train or evaluate. |
| Subvillage heads (`RTHEAD == 1`) | 640 | Drop: added on purpose, not sampled, and better off (27% vs 36% poor). |
| `se2_139` coded 1/3 instead of 0/1 | all | Recode 3 → 0. |
| Head education missing (all 4 `hheduc` = 0) | 8 | Otherwise silently read as "no school" (`hheduc1`, the reference). Add an indicator `hheduc_missing`, then drop `hheduc1`. |
| `depratio` missing | 228 | All are households with no member aged 15–64 (division by zero). Add `no_workingage`; filling happens in the pipeline. |
| `pcfloor` missing | 52 | Add `pcfloor_missing`; median filled in the pipeline (training folds only). |
| A few missing 0/1 dummies (`toilet` 8, `se2_122_123` 4, `water`, `se5_110`, …) | ≤ 8 each | Leave missing; most frequent value filled in the pipeline. |
| `hr06xx` count missing because the item is not owned (dummy = 0) | most rows | Fill with 0: "not owned" means 0 items, by definition (no learning from data, so no leakage). |
| `hr06xx` count missing although the dummy says owned | 61 jewelry, 1 each motorcycle, car, chicken, pig | Leave missing (0 would contradict the dummy); filled in the pipeline. |
| `pcexppred` < 0 | 149 | Keep. `PMTSCORE = log(pcexppred)`, so these are exactly the 149 missing `PMTSCORE`. A negative prediction still ranks the household as poorest, which is what the baseline needs. |
| Households exactly on the poverty line | 6 (2 coded poor, 4 not) | Use `poor` as given; do not rebuild it with `<`. |

**Unusual but plausible: keep (lecture 106: outlier treatment depends on the task)**
- `hhage` of 14 and 105 (2 households): possible for a household head; no reason to treat as an error.
- `pcfloor` > 200 m² per person (22 households): mostly single-person households with a large house (e.g. 250 m², 1 person). No outliers on the log scale (Tukey k = 3); Yeo–Johnson in the pipeline handles the skew.
- 2 Tukey outliers (k = 3) on log consumption: real households; the log target and the ranking-based metric already limit their influence. Removing target outliers would be cherry-picking.

</details>

## Merge before cleaning

In the Big Assignment we cleaned each dataset separately and merged at the end (Q3 → Q5),
because the datasets were separate tables that only needed to be joined for the analysis.
Here the situation is different, so we merge **before** any cleaning:

- **The two files are two parts of the same household record.** `coding_suseti_pmt` holds the
  survey and the government's variables, `mistargeting_CORRECTED` holds the poverty label and
  the subvillage-head flag. Both contain the same 5,756 households, one row each, linked by
  `hhid = hhid_suseti` (checked unique in step 1).
- **The main cleaning decisions need columns from both files.** Dropping the subvillage heads
  uses `RTHEAD` (labels), dropping households without a target uses `LOGCONSUMPTION` (main)
  and `poor` (labels). Cleaning the files separately would mean applying every row filter
  twice and checking afterwards that both files still contain the same households.
- **Merging first makes lost rows visible.** We merge with `how="outer"`, `indicator=True` and
  `validate="one_to_one"`: every household is matched (5,756 in both, none in only one file)
  and no row is duplicated. A merge at the end of the cleaning would hide mistakes: an inner
  merge silently drops any household that one file has lost on the way.
- **Merging does not change any values.** It only puts the columns side by side, so the
  diagnosis above (done on the separate files) still holds for the merged data.

After the merge, `hhid_suseti` is dropped (identical to `hhid`), and the cleaning works on one
dataset, `df`.


In [6]:
# 6. Merge the two files (BA Q5) before cleaning.
#    Both describe the same households one-to-one; the keys were checked in step 1.

# --- 1. No column apart from the key exists in both files (otherwise pandas would add _x / _y)
overlap = (set(main.columns) & set(labels.columns)) - {"hhid", "hhid_suseti"}
assert not overlap, f"columns in both files: {overlap}"

# --- 2. Merge: outer + indicator shows any household that is in only one file,
#        validate stops if a key is duplicated (one-to-many would multiply rows)
households = main.merge(labels, left_on="hhid", right_on="hhid_suseti",
                        how="outer", validate="one_to_one", indicator=True)
print("merge result:", households["_merge"].value_counts().to_dict())
assert (households["_merge"] == "both").all(), "a household is in only one file"

# --- 3. Tidy up: the indicator and the second key are no longer needed, drop them
assert (households["hhid"] == households["hhid_suseti"]).all()
households = households.drop(columns=["_merge", "hhid_suseti"])

# --- 4. Verify: same households, all columns of both files
assert len(households) == len(main) == len(labels) == 5756
assert households.shape[1] == main.shape[1] + labels.shape[1] - 1
print("merged:", households.shape)


merge result: {'both': 5756, 'left_only': 0, 'right_only': 0}
merged: (5756, 113)


In [7]:
# 6. Clean: act on each finding from the diagnosis, one at a time (BA Q3.2).

# --- 6.1 Villages with 8 instead of 9 households: keep, no rows changed.
#     They are complete, valid observations (one head, one share value each);
#     a smaller sample is not an error, so there is nothing to fix or impute. -> only decision rule later changes with (village size x share and not 8 x share)
hh_per_village = main.groupby("hhea").size()
small_villages = hh_per_village[hh_per_village < 9].index.tolist()
print("villages with fewer than 9 households:", small_villages) #this shows the finding with the villages fewer than 9

assert hh_per_village.between(8, 9).all(), "a village has fewer than 8 or more than 9 households" #this as double check if data were changed later so that it would still work
assert main.groupby("hhea")["share"].nunique().eq(1).all(), "share differs within a village" #this makes sure that the household shares are the same for everybody in the same village - also doublecheck - maybe later delete


# --- 7.1 Drop the subvillage heads (RTHEAD == 1)
# why: they were added to each village on purpose, not drawn at random like the other 8,
#      and are better off (27% poor vs. 36%). They would be 11% of our sample but only ~2%
#      of a real village, so keeping them trains and evaluates on a population that does not exist.
n_before = len(households)
households = households[households["RTHEAD"] == 0].drop(columns="RTHEAD")
# why drop the column too: it was kept only for this filter; after it, it is 0 everywhere
print(f"7.1 dropped {n_before - len(households)} subvillage heads -> {len(households)} households")
assert n_before - len(households) == 640

# --- 7.2 Drop households without a target (LOGCONSUMPTION and poor missing)
# why: a row without a target can be used neither to train nor to evaluate. Imputing it would
#      invent the answer the model should learn. Not leakage: the rule is fixed in advance and
#      learns nothing from the data, so it may run before the train/test split.
missing_target = households["LOGCONSUMPTION"].isna() | households["poor"].isna()
assert (households["LOGCONSUMPTION"].isna() == households["poor"].isna()).all()   # why: both missing for the same households
n_before = len(households)
households = households[~missing_target]
print(f"7.2 dropped {n_before - len(households)} households without a target -> {len(households)} households")
assert len(households) == 5113

# --- 7.3 Recode se2_139 (non-agricultural land) from 1/3 to 1/0
# why: the survey codes yes/no as 1 = "Ya" (yes) / 3 = "Tidak" (no). Every other asset dummy
#      is 0/1; left as 1/3, a linear model would read "no" as 3x "yes".
print("7.3 se2_139 before:", households["se2_139"].value_counts(dropna=False).to_dict()) #gives value coutns without dropping NA - here gives 3 as value
households["se2_139"] = households["se2_139"].replace({3: 0}) #replaces 3 with 0 so dummy consistent with rest
print("    se2_139 after: ", households["se2_139"].value_counts(dropna=False).to_dict())
assert set(households["se2_139"].dropna().unique()) <= {0, 1}
# why the missing values stay: "not answered" is not "no"; they are filled in the pipeline
#      (most frequent value, fitted on the training folds only)

# --- Check: households per village after both drops
# why: the quota uses the actual village size (step 7.0), so the sizes must be as expected
print("households per village:", households.groupby("hhea").size().value_counts().to_dict())   # {8: 633, 7: 7} so 633x 8 and 7x 7
assert households.groupby("hhea").size().between(7, 8).all()

# General rule for missing values in this step:
# why: a missing value must never silently become 0 (= "no", "none", or the reference category).
#      Here we only (a) add 0/1 indicators where "missing" itself carries information and
#      (b) leave the values missing (<NA>). Filling (median / most frequent) is learned from
#      the data, so it happens in the pipeline on the training folds only (no leakage).

# --- 7.4 Head education missing (all 4 hheduc dummies = 0, 8 households)
educ = ["hheduc1", "hheduc2", "hheduc3", "hheduc4"]
households["hheduc_missing"] = (households[educ].sum(axis=1) == 0).astype("Int64")
# why: with hheduc1 as reference, "all zeros" would read as "no school" although it is unknown;
#      the indicator gives these heads their own category
households = households.drop(columns="hheduc1")
# why drop hheduc1: the education dummies always sum to 1, so one must be left out as the
#      reference (dummy trap); the government formula also uses hheduc2-4 only
print("7.4 heads with missing education:", int(households["hheduc_missing"].sum()))
assert (households[["hheduc2", "hheduc3", "hheduc4", "hheduc_missing"]].sum(axis=1) <= 1).all()

# --- 7.5 depratio missing = no member aged 15-64 (220 households)
assert (households["depratio"].isna() == (households["age1564"] == 0)).all()
# why: confirms every missing depratio is a division by zero, not a missing answer
households["no_workingage"] = (households["age1564"] == 0).astype("Int64")
# why: "nobody of working age" is real information (mostly elderly living alone); depratio stays
#      missing and is filled in the pipeline, the indicator keeps the information
households = households.drop(columns="age1564")
# why drop age1564: helper column, only needed to build this indicator, this info is already contained in hhsize and depretio column and is not in gov formula for set A
print("7.5 households without working-age members:", int(households["no_workingage"].sum()))

# --- 7.6 pcfloor (floor area per person) missing (51 households)
households["pcfloor_missing"] = households["pcfloor"].isna().astype("Int64") #this adds the new column
# why: missingness is informative (45% poor vs. 36% overall); pcfloor itself stays missing and
#      gets the median in the pipeline later (median: pcfloor is very skewed, mean 24 vs. median 17 m²)
print("7.6 households with missing pcfloor:", int(households["pcfloor_missing"].sum()))

# --- 7.7 Few missing 0/1 dummies (<= 13 per column): leave as <NA>
# why: "not answered" is not "no", so setting them to 0 would be wrong; too few for an indicator
#      to carry information; filled with the most frequent value in the pipeline
dummy_na = households[binary_cols_after := [c for c in binary if c in households.columns]].isna().sum()
print("7.7 missing 0/1 dummies left as <NA>:", dummy_na[dummy_na > 0].to_dict())

# --- 7.8 Asset counts (hr06xx): fill with 0 only where the dummy says "not owned"
# why: if the household does not own the item (dummy = 0), the number owned is 0 by definition.
#      This is a fixed rule, not learned from the data, so no leakage and it may run before the split.
assert set(count_to_dummy) == set(set_b_counts), "every count needs its dummy"
# why: hr0607/hr0608 were excluded in step 2 (no label, no dummy), so all remaining counts have one

filled = {}
for cnt, dum in count_to_dummy.items():
    not_owned = households[cnt].isna() & (households[dum] == 0).fillna(False)
    # why fillna(False): where the dummy itself is missing we do not know it is "not owned"
    households.loc[not_owned, cnt] = 0
    filled[cnt] = int(not_owned.sum())
print(f"7.8 hr06xx: filled {sum(filled.values())} cells with 0 (item not owned)")

# why the rest stays <NA>: dummy = 1 but count missing -> owned, number unknown (0 would contradict
#      the dummy); dummy missing -> ownership unknown. Both are filled in the pipeline (training folds only).
still_na = households[set_b_counts].isna().sum()
print("    still missing:", still_na[still_na > 0].to_dict())   # expected: jewelry 51+, motorcycle, chicken, pig, + 8 with missing dummy

# check: no count contradicts its dummy after filling
for cnt, dum in count_to_dummy.items():
    assert not ((households[cnt] > 0) & (households[dum] == 0)).fillna(False).any(), f"{cnt} > 0 but {dum} = 0"
    assert not ((households[cnt] == 0) & (households[dum] == 1)).fillna(False).any(), f"{cnt} = 0 but {dum} = 1"

# --- 7.9 pcexppred < 0 (141 households): keep as it is
# why: pcexppred is only the government baseline, never a feature, and the baseline only needs the
#      ranking: a negative predicted consumption ranks the household as poorest, as the formula intends
#      (77% of them are poor). Taking the log (PMTSCORE) would turn exactly these into missing values.
print("7.9 negative pcexppred kept:", int((households["pcexppred"] < 0).sum()))

# --- 7.10 Households exactly on the poverty line (6): use poor as given
# why: poor comes from the data provider; rebuilding it with "consumption < line" would recode
#      2 households and change the target. The check below confirms poor = "below the line" everywhere else.
on_line = np.isclose(np.exp(households["LOGCONSUMPTION"]), households["povline_poor"], rtol=1e-6)
below = np.exp(households["LOGCONSUMPTION"]) < households["povline_poor"]
assert (below[~on_line].astype(int) == households.loc[~on_line, "poor"]).all()
print(f"7.10 on the poverty line: {on_line.sum()} households, {int(households.loc[on_line, 'poor'].sum())} coded poor (kept as given)")

households = households.drop(columns="povline_poor")
# why drop povline_poor: kept only for this check; it is not a feature and not needed later



villages with fewer than 9 households: [67, 612, 617, 622]
7.1 dropped 640 subvillage heads -> 5116 households
7.2 dropped 3 households without a target -> 5113 households
7.3 se2_139 before: {np.int64(3): 4654, np.int64(1): 446, <NA>: 13}
    se2_139 after:  {np.int64(0): 4654, np.int64(1): 446, <NA>: 13}
households per village: {8: 633, 7: 7}
7.4 heads with missing education: 8
7.5 households without working-age members: 220
7.6 households with missing pcfloor: 51
7.7 missing 0/1 dummies left as <NA>: {'toilet': 8, 'water': 1, 'se2_122_123': 4, 'se5_110': 1, 'se2_112': 1, 'se2_121': 1, 'se5_17': 1, 'se2_139': 13}
7.8 hr06xx: filled 97393 cells with 0 (item not owned)
    still missing: {'hr0623': 1, 'hr0627': 55, 'hr0628': 2, 'hr0629': 2, 'hr0634': 1, 'hr0635': 1}
7.9 negative pcexppred kept: 141
7.10 on the poverty line: 6 households, 2 coded poor (kept as given)


In [8]:
# 8. Final checks and save

# --- shape before and after cleaning
print("raw:", main.shape, "households | clean:", households.shape) #the new columns are the dummies that were added

# --- save
os.makedirs("../data/processed", exist_ok=True)
households.to_pickle("../data/processed/households_clean.pkl")
# why pickle, not CSV: it keeps the dtypes (Int64 with <NA>, hhid as text with leading zeros "001025");
#      a CSV would read hhid back as a number and lose the zeros

raw: (5756, 110) households | clean: (5113, 112)


In [9]:
# 9. Test villages and CV folds (outline section 4; notebook 108b)
SEED = 40   # outline: random_state=40 everywhere

# --- 9.1 One row per village: the unit we split
# why villages, not households: the formula is applied to new villages, and households in one
#      village share prices, roads and shocks. A household split would put neighbours into
#      training and test, so the test score would be too optimistic (same problem as the
#      random split of hourly bike data in 108b).
villages = households.groupby("hhea").agg(kabu=("kabu", "first")).reset_index()
assert households.groupby("hhea")["kabu"].nunique().eq(1).all()   # why: every village lies in one district

# --- 9.2 Test set: 20% of the villages, drawn within each district (train_test_split as in class)
# why stratify=kabu: every district, and so every regional government formula, appears in training and test
train_villages, test_villages = train_test_split(
    villages, test_size=0.2, stratify=villages["kabu"], random_state=SEED)
is_test = households["hhea"].isin(test_villages["hhea"])
print(f"train: {len(train_villages)} villages, {(~is_test).sum()} households | "
      f"test: {len(test_villages)} villages, {is_test.sum()} households")
assert not set(train_villages["hhea"]) & set(test_villages["hhea"])   # why: no village in both parts

# --- 9.3 5 CV folds on the TRAINING villages only (KFold as in class, stratified by district)
# why only training villages: the test villages are used once at the very end, never for choosing
# why shuffle: otherwise the folds follow the row order (villages sorted by district), as noted in 108b
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
train_villages = train_villages.reset_index(drop=True)   # why: positions 0..511, so the fold indices match the rows
train_villages["fold"] = -1                               # why: create the column first; -1 = "no fold yet"
for fold, (_, val_idx) in enumerate(cv.split(train_villages, train_villages["kabu"])):
    train_villages.loc[val_idx, "fold"] = fold            # each village is the validation fold exactly once
assert (train_villages["fold"] >= 0).all()                # why: every training village got a fold

# --- 9.4 Balance check (outline: share poor, urban share, province mix)
check = households.assign(part=np.where(is_test, "test", "train"))
print(check.groupby("part")[["poor", "urban"]].mean().astype(float).round(3))
print(pd.crosstab(check["part"], check["prop"], normalize="index").round(3))
folds = households.merge(train_villages[["hhea", "fold"]], on="hhea")
print(folds.groupby("fold").agg(households=("hhid", "size"), villages=("hhea", "nunique"),
                                poor=("poor", "mean")).astype(float).round(3))

# --- 9.5 Save, so every model uses identical test villages and folds
# why per village, not per household: a village can never end up in two folds
test_villages[["hhea"]].sort_values("hhea").to_csv("../data/processed/test_villages.csv", index=False)
train_villages[["hhea", "fold"]].sort_values("hhea").to_csv("../data/processed/cv_folds.csv", index=False)


train: 512 villages, 4090 households | test: 128 villages, 1023 households
        poor  urban
part               
test   0.367  0.492
train  0.363  0.490
prop      12     33     73
part                      
test   0.249  0.461  0.289
train  0.254  0.450  0.296
      households  villages   poor
fold                             
0          822.0     103.0  0.364
1          822.0     103.0  0.387
2          815.0     102.0  0.369
3          815.0     102.0  0.353
4          816.0     102.0  0.339


## Column selection: which columns we keep and why

<details>
<summary><b>Click to open: how we decided which columns to keep or drop</b></summary>

The raw files have 347 (`main`) and 95 (`labels`) columns. We reduce them **before merging** (same order as Q2 → Q5 of the Big Assignment solution). The code lists what we **keep**, not what we drop: any column not listed is removed, so a leakage column cannot slip through because someone forgot it in a drop list. If we later need a dropped column, we add it to the keep list and rerun the notebook.

### Questions asked of every column, in order (the first "yes" decides)

**1. Is it consumption or derived from it? → drop (direct leakage).**
It contains the answer we want to predict. Consumption is never an input: it is too costly to measure for every household, which is the whole reason the government uses a proxy-means test.
- Dropped: `hhconsumption`, `wfood`, `mfood`, `mnonfood`, `CONSUMPTION`, `RANKCONSUMPTION`, `rank_pcexp`, `nhhrank_pcexp`, `lnpcexpe`, `epcexpe`, `avg_CONSUMPTION`, `povertygap`, `verypoor`, `nearpoor`, `middle`, `rich`, `poorhc`, `verypoorhc`, `middlehc`.
- Exception: `LOGCONSUMPTION` and `poor` are kept, because they are our **targets** (never features).

**2. Is it only known after the cutoff? → drop (leakage from the future).**
Our cutoff is the door-to-door asset survey, *before* allocation. Anything describing who was selected, how the experiment was run or how the targeting turned out is the *result* of the selection, and it mixes the experiment's three targeting methods (which would move us towards a causal question we don't ask).
- Dropped: `RTS`, `RTS_meeting`, `ranking*`, `nhhrank*`, `TREATRANK`, `MISTARGET*`, `maintreatment` and all `*subtreatment`, `PMT`, `COMMUNITY`, `HYBRID`, `ELITE*`, `meeting_order`, `facilitator_code`, `fund_order`, `village_order`, `quota*`.
- Exception: `share`, the village quota fixed **before** the experiment, is kept: the decision rule needs it (top `round(8 × share)` households per village). It is never a feature.

**3. Is it the government's score or part of it? → drop, except one baseline column.**
The model must not learn from the government's prediction or its pieces; otherwise "beating the formula" would partly be copying it.
- Dropped: `PMTSCORE`, `PMTRANK`, `rank_pmt`, `nhhrank_pmt`, all `c_*` (formula coefficients), `comp_*`, `comp2_*`, `*_avg`, `*_dm`, `temp*`.
- Exception: `pcexppred` is kept **only as the government-formula baseline**, never as a feature. (`PMTSCORE` is not used: it is missing for 149 households, and `pcexppred` gives the same ranking.)

**4. Is it self-reported or a social item? → drop, unless it is in the government formula.**
In a real programme survey people know their answers can decide who gets money, so self-reports are easy to bend. Social items (connections, leaders, neighbours' views) are not something an enumerator observes at the door.
- Dropped: `healthcare` ("can afford healthcare"), `shock`, `ETHICMINORITY`, `RELIGIOUSMINORITY`, `elite_connectedness`, `FAMILY*`, `say_*`, `*LEADER`.
- Exception: `credit` (ever received a loan) is self-reported **but kept**, because it is in the government formula. Our rule for set A is "the same information as the government"; dropping it would make the comparison unfair.

**5. Is it in the government formula? → keep as set A (main feature set).**
The raw file contains the government's coefficients (`c_*` columns), so we read the formula variables directly from the data: **62 variables** (dwelling, household, head, household education, credit, urban, village census, assets).
- Note: `se5_15_16` (goat/sheep) and `se2_138` (agricultural land) have coefficients in the formula, so they belong to **set A**, not set B as first written in our outline.

**6. Can an enumerator see or verify it at the door? → keep as set B (extension).**
Extra assets (rice cooker, mixer, fan, telephone, sewing machine, water pump, rice husker, boats), livestock (pigs, horses), non-agricultural land, floor quality (`FLOORTOPQUALITY`) and the `hr06xx` asset counts.
- `se2_110` (fan), `se5_13` (horse) and `se2_139` (non-agricultural land) have a `c_*` column, but it is empty: they are **not** in the formula, so they are set B.

**7. Needed to identify, merge, filter, split or apply the quota? → keep, with a stated reason.**

| Column | Why we keep it |
|---|---|
| `hhid`, `hhid_suseti` | Merge key (store as text: leading zeros, e.g. `"001025"`) |
| `hhea` | Village: quota groups, train/test split, CV groups |
| `kabu` | District: district dummies; test villages are drawn within each district |
| `prop` | Province: balance check, province hold-out |
| `share` | Village quota for the decision rule |
| `pcexppred` | Government-formula baseline |
| `LOGCONSUMPTION`, `poor` | Targets |
| `povline_poor` | Only to document/check `poor` |
| `RTHEAD` | Only to drop the subvillage heads, then removed |
| `age1564` | Only to build `no_workingage`, then removed |

`kab` is dropped because `kabu` already contains it.

**8. Anything else → drop.**

### Village-level variables stay in the clean data
The 14 inputs that are the same for every household in a village (11 village facilities, province, district, `urban`) cannot change the ranking *within* a village. Following the feedback, the **linear models** will not use them, but **boosting and the neural net** keep them (they can combine them with household features). So they stay in the clean file; each model chooses its columns later in `src/features.py`, not here in the cleaning.

### Open points to discuss in the group
- Agree that `se5_15_16` and `se2_138` move to set A (update the outline).
- Agree to keep `credit` despite being self-reported (in the formula).
- Do we need `povline_poor` in the clean file, or only for the check here?

</details>